In [ ]:
# ─────────────────────────────────────────────────────
# P3 – CINEMATICA INVERSA 
# ─────────────────────────────────────────────────────
import sys
import os
import numpy as np
import pandas as pd
import time
import math
from typing import List

# Agregar rutas de módulos P1 y P2
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'P1_DH'))
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'P2_FK'))

print('Librerías cargadas ✓')

# ─────────────────────────────────────────────────────
# CONEXION AL ROBOT
# ─────────────────────────────────────────────────────
from pymycobot.mycobot import MyCobot

mc = MyCobot('/dev/ttyUSB0', 1000000)
mc.power_on()
time.sleep(1)
assert mc.is_controller_connected(), 'Error de conexión'
print('Robot conectado ✓')

## Importar módulos de cinemática P1, P2 y P3

In [ ]:
# Importar módulos de P1, P2 y P3
try:
    from p1_dh_representacion import DH_TABLE, forward_kinematics, extract_position
    from p2_cinematica_directa import ForwardKinematics
    from p3_cinematica_inversa import InverseKinematics, compare_ik_solutions
    print('Módulos P1, P2, P3 importados ✓')
except ImportError as e:
    print(f'[ERROR] No se pudieron importar módulos: {e}')
    raise

# ─────────────────────────────────────────────────────
# TABLA DE PARAMETROS DH (referencia de P1)
# ─────────────────────────────────────────────────────
DH_TABLE_LOCAL = [
    (   0,   134.75,  90.0,   0.0),   # J1 – Base
    (-110,     0.0,   0.0,  -90.0),   # J2 – Hombro
    ( -96,     0.0,   0.0,    0.0),   # J3 – Codo
    (   0,    63.40,  90.0,  -90.0),  # J4 – Muñeca 1
    (   0,    75.05, -90.0,   90.0),  # J5 – Muñeca 2
    (   0,    50.00,   0.0,    0.0),  # J6 – Gripper
]

JOINT_NAMES = ['J1 Base', 'J2 Hombro', 'J3 Codo', 'J4 Muñeca1', 'J5 Muñeca2', 'J6 Gripper']
JOINT_LIMITS = [
    (-168, 168), (-135, 90), (-150, 150),
    (-145, 145), (-165, 165), (-180, 180),
]

## Verificación inicial en HOME

In [ ]:
# ─────────────────────────────────────────────────────
# VERIFICACION CON FK EN POSE HOME (referencia de P2)
# ─────────────────────────────────────────────────────

# Funciones de FK (copiadas de P2 para referencia)
def dh_matrix(theta_deg, d, a, alpha_deg):
    """Matriz de transformación homogénea DH 4x4 para un eslabón."""
    from math import radians, cos, sin
    theta = radians(theta_deg)
    alpha = radians(alpha_deg)
    ct, st = cos(theta), sin(theta)
    ca, sa = cos(alpha), sin(alpha)
    return np.array([
        [ct, -st*ca,  st*sa, a*ct],
        [st,  ct*ca, -ct*sa, a*st],
        [ 0,     sa,     ca,    d],
        [ 0,      0,      0,    1],
    ], dtype=float)

def build_Ti_list(thetas_deg):
    """Devuelve lista de 6 matrices Ti para los ángulos dados."""
    return [
        dh_matrix(thetas_deg[i] + DH_TABLE_LOCAL[i][3],
                  DH_TABLE_LOCAL[i][1],
                  DH_TABLE_LOCAL[i][0],
                  DH_TABLE_LOCAL[i][2])
        for i in range(6)
    ]

def forward_kinematics(thetas_deg):
    """Devuelve T06 (4x4). Posición del extremo = T06[0:3, 3]"""
    T = np.eye(4)
    for Ti in build_Ti_list(thetas_deg):
        T = T @ Ti
    return T

def get_xyz(T06):
    """Extrae la posición (x, y, z) de la matriz T06"""
    return T06[0, 3], T06[1, 3], T06[2, 3]

# Mover a HOME y leer posición real
print("Moviendo a HOME...")
mc.send_angles([0, 0, 0, 0, 0, 0], 25)
time.sleep(4)
coords_real = mc.get_coords()   # [x, y, z, rx, ry, rz]

# FK calculada en HOME
T06_home = forward_kinematics([0, 0, 0, 0, 0, 0])
x_c, y_c, z_c = get_xyz(T06_home)

# Comparación
error = np.linalg.norm([x_c - coords_real[0],
                        y_c - coords_real[1],
                        z_c - coords_real[2]])

print(f'\nPosición real  (mc.get_coords()): x={coords_real[0]:8.2f}, y={coords_real[1]:8.2f}, z={coords_real[2]:8.2f} mm')
print(f'Posición FK calculada:            x={x_c:8.2f}, y={y_c:8.2f}, z={z_c:8.2f} mm')
print(f'\nError euclidiano: {error:.2f} mm')
print(f'  Δx = {abs(x_c - coords_real[0]):.2f} mm')
print(f'  Δy = {abs(y_c - coords_real[1]):.2f} mm')
print(f'  Δz = {abs(z_c - coords_real[2]):.2f} mm')
print(f'\nMatriz T06 en HOME:')
print(np.round(T06_home, 4))

## Pruebas de IK analítica para varias posiciones

In [ ]:
# ─────────────────────────────────────────────────────
# PRUEBAS DE IK ANALITICA (P3)
# ─────────────────────────────────────────────────────

ik = InverseKinematics()

# 5 posiciones de prueba
test_positions = [
    (150,   0, 200, "Frente al robot"),
    (100, 100, 180, "Diagonal derecha"),
    (100,-100, 200, "Diagonal izquierda"),
    ( 80,  80, 220, "Lateral izquierdo elevado"),
    (120,   0, 250, "Arriba al frente"),
]

print("\n" + "="*100)
print("P3 - IK ANALITICA (resolver_pdf): PRUEBAS PARA 5 POSICIONES")
print("="*100)

ik_results = []
for x, y, z, desc in test_positions:
    print(f"\n[{desc}]  x={x:6.1f}, y={y:6.1f}, z={z:6.1f} mm")
    try:
        # IK analítica con codo arriba
        angles = ik.resolver_pdf(x, y, z, codo_arriba=True)
        print(f"  IK Analítica (codo arriba):  {[round(a, 2) for a in angles]}")
        ik_results.append({
            'Posición': desc,
            'x': x, 'y': y, 'z': z,
            'IK_analitica': angles,
            'Status': 'OK'
        })
    except ValueError as e:
        print(f"  IK Analítica: FALLO - {e}")
        ik_results.append({
            'Posición': desc,
            'x': x, 'y': y, 'z': z,
            'IK_analitica': None,
            'Status': 'FUERA DE WORKSPACE'
        })

print("\n" + "="*100)

## Comparación IK analítica vs API

In [ ]:
# ─────────────────────────────────────────────────────
# COMPARACION: IK ANALITICA vs API
# ─────────────────────────────────────────────────────

print("\n\n" + "="*100)
print("P3 - COMPARACION IK ANALITICA vs API (resolver_via_api)")
print("TABULAR EL ERROR EN GRADOS ENTRE AMBAS SOLUCIONES (como pide el examen)")
print("="*100)

# Ejecutar la función de comparación completa
results_comparison = compare_ik_solutions(mc=mc)

print("\nResumen de comparación:")
for i, res in enumerate(results_comparison, 1):
    print(f"\nPosición {i}: {res['xyz']}")
    if res.get('analitica'):
        print(f"  Solución IK analítica: {[round(a, 2) for a in res['analitica']]}")
    if res.get('api'):
        print(f"  Solución API:          {[round(a, 2) for a in res['api']]}")
    if res.get('error_grados'):
        print(f"  Error (grados):        {res['error_grados']}")
    if res.get('error'):
        print(f"  Error: {res['error']}")

##  Verificación FK de las soluciones IK (validar cierre del loop)

In [ ]:
# ─────────────────────────────────────────────────────
# VERIFICACION FK: Aplicar FK a los ángulos IK obtenidos
# Válida que el cierre del loop IK→FK da la posición esperada
# ─────────────────────────────────────────────────────

print("\n" + "="*100)
print("VERIFICACION FK DE SOLUCIONES IK")
print("Aplicar FK a los ángulos calculados por IK y verificar que recuperamos la posición")
print("="*100)

for i, (x, y, z, desc) in enumerate(test_positions[:3], 1):  # usar primeras 3 posiciones
    print(f"\n[Posición {i}: {desc}] (x, y, z) = ({x}, {y}, {z}) mm")
    
    if ik_results[i-1]['IK_analitica']:
        angles_ik = ik_results[i-1]['IK_analitica']
        T06_calc = forward_kinematics(angles_ik)
        x_calc, y_calc, z_calc = get_xyz(T06_calc)
        
        error_xyz = np.linalg.norm([x_calc - x, y_calc - y, z_calc - z])
        
        print(f"  Ángulos IK:          {[round(a, 2) for a in angles_ik]}")
        print(f"  Pos esperada:        ({x:.2f}, {y:.2f}, {z:.2f}) mm")
        print(f"  Pos FK calculada:    ({x_calc:.2f}, {y_calc:.2f}, {z_calc:.2f}) mm")
        print(f"  Error de cierre:     {error_xyz:.2f} mm")
        
        if error_xyz < 1.0:
            print(f"  ✓ CIERRE CORRECTO (error < 1 mm)")
        elif error_xyz < 5.0:
            print(f"  ⚠ CIERRE ACEPTABLE (error < 5 mm)")
        else:
            print(f"  ✗ CIERRE POBRE (error > 5 mm)")

print("\n" + "="*100)

## Tabla resumen y análisis de configuraciones singulares

In [ ]:
# ─────────────────────────────────────────────────────
# TABLA RESUMEN Y SINGULARIDADES
# ─────────────────────────────────────────────────────

# Tabla resumen de IK analítica
print("\n" + "="*100)
print("TABLA RESUMEN: IK ANALITICA PARA 5 POSICIONES")
print("="*100)

df_summary = pd.DataFrame(ik_results)
print(df_summary[['Posición', 'x', 'y', 'z', 'Status']].to_string(index=False))

print("\n" + "="*100)
print("CONFIGURACIONES SINGULARES DEL MyCobot 280")
print("="*100)

singularidades = [
    ("Brazo completamente extendido", "J1=0, J2=-20, J3=0",
     "Codo alineado → múltiples soluciones J2/J3"),
    ("Singularidad overhead (eje Z)", "x≈0, y≈0 (brazo apunta verticalmente)",
     "J1 indefinido (atan2(0,0)) → singularidad de base"),
    ("Muñeca alineada", "J5=0°",
     "J4 y J6 colineales → perdemos 1 DOF de orientación"),
    ("Retracción máxima", "J3=-150° o J3=+150°",
     "Brazo completamente plegado → espacio reducido"),
]

for i, (nombre, config, desc) in enumerate(singularidades, 1):
    print(f"\n[Singularidad {i}] {nombre}")
    print(f"  Config:  {config}")
    print(f"  Efecto:  {desc}")

print("\n" + "="*100)
print("NOTAS IMPORTANTES")
print("="*100)
print("""
1. La solución IK analítica usa solo J1, J2, J3 (primeros 3 joints)
   → J4, J5 quedan en pose fija (0, 0)
   → J6 se fija en -45° (giro del gripper)

2. La API del robot (mc.send_coords) usa IK numérica interna 
   → puede encontrar soluciones distintas de la analítica
   → el error entre ambas debe ser < 10° para considerar validez

3. El "cierre del loop" (FK de ángulos IK) debe verificar:
   - Error XYZ < 1 mm ✓ excelente
   - Error XYZ < 5 mm ⚠ aceptable
   - Error XYZ > 5 mm ✗ problema

4. Para usar IK 6DOF con scipy.optimize:
   from p3_cinematica_inversa import InverseKinematics
   ik = InverseKinematics()
   angles_6dof = ik.resolver_6dof(x, y, z, rx, ry, rz, solo_posicion=False)
""")
print("="*100)

print("\n✓ TEST IK COMPLETADO\n")